In [4]:
import os
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

import tensorflow as tf

In [6]:
DATA_DIR = "../data/manual_csv"

data = []
labels = []

for file in os.listdir(DATA_DIR):
    if file.endswith(".csv"):
        path = os.path.join(DATA_DIR, file)
        df = pd.read_csv(path, header=None)
        data.append(df.iloc[:, :-1])
        labels.append(df.iloc[:, -1])

X = pd.concat(data).values
y = pd.concat(labels).values

X.shape, y.shape

((158450, 126), (158450,))

In [7]:
encoder = LabelEncoder()
y_encoded = encoder.fit_transform(y)

encoder.classes_


array(['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M',
       'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z'],
      dtype=object)

In [ ]:
model = Sequential([
    tf.keras.Input(shape=(X_train.shape[1],)),
    Dense(128, activation="relu"),
    Dense(64, activation="relu"),
    Dense(len(encoder.classes_), activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()


Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 128)               8192      
                                                                 
 dense_1 (Dense)             (None, 64)                8256      
                                                                 
 dense_2 (Dense)             (None, 5)                 325       
                                                                 
Total params: 16,773
Trainable params: 16,773
Non-trainable params: 0
_________________________________________________________________


In [ ]:
checkpoint_path = os.path.join("../model", "best_gesture_model.h5")
os.makedirs(os.path.dirname(checkpoint_path), exist_ok=True)

callbacks = [
    EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    ModelCheckpoint(checkpoint_path, monitor="val_loss", save_best_only=True)
]



history = model.fit(
    X_train,
    y_train,
    validation_data=(X_test, y_test),
    epochs=50,
    batch_size=32,
    callbacks=callbacks
)

Epoch 1/25
25/25 [==============================] - 2s 19ms/step - loss: 1.3684 - accuracy: 0.6150 - val_loss: 1.1640 - val_accuracy: 1.0000
Epoch 2/25
25/25 [==============================] - 0s 4ms/step - loss: 0.9708 - accuracy: 0.8975 - val_loss: 0.8111 - val_accuracy: 1.0000
Epoch 3/25
25/25 [==============================] - 0s 4ms/step - loss: 0.6305 - accuracy: 1.0000 - val_loss: 0.5324 - val_accuracy: 1.0000
Epoch 4/25
25/25 [==============================] - 0s 5ms/step - loss: 0.4033 - accuracy: 1.0000 - val_loss: 0.3362 - val_accuracy: 1.0000
Epoch 5/25
25/25 [==============================] - 0s 4ms/step - loss: 0.2505 - accuracy: 1.0000 - val_loss: 0.2011 - val_accuracy: 1.0000
Epoch 6/25
25/25 [==============================] - 0s 4ms/step - loss: 0.1565 - accuracy: 1.0000 - val_loss: 0.1260 - val_accuracy: 1.0000
Epoch 7/25
25/25 [==============================] - 0s 5ms/step - loss: 0.0986 - accuracy: 1.0000 - val_loss: 0.0813 - val_accuracy: 1.0000
Epoch 8/25
25/25 [=

In [9]:
import os
import pickle

MODEL_DIR = "../model"
os.makedirs(MODEL_DIR, exist_ok=True)

# Save trained model
model.save(os.path.join(MODEL_DIR, "gesture_model.h5"))

# Save label encoder
with open(os.path.join(MODEL_DIR, "label_encoder.pkl"), "wb") as f:
    pickle.dump(encoder, f)


In [31]:
import os

DATASET_PATH = "../data/ISL"

print("Folders found:")
print(sorted(os.listdir(DATASET_PATH)))

Folders found:
['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z']


In [32]:
import os
import cv2
import mediapipe as mp
import pandas as pd

# -----------------------------
# Paths
# -----------------------------
DATASET_PATH = "../data/ISL"
OUTPUT_CSV = "../data/isl_landmarks.csv"

# -----------------------------
# MediaPipe
# -----------------------------
mp_hands = mp.solutions.hands

hands = mp_hands.Hands(
    static_image_mode=True,
    max_num_hands=1,
    min_detection_confidence=0.5
)

# -----------------------------
# Storage
# -----------------------------
rows = []
failed = []

# -----------------------------
# Loop through A-Z folders
# -----------------------------
for label in sorted(os.listdir(DATASET_PATH)):

    label_path = os.path.join(DATASET_PATH, label)

    if not os.path.isdir(label_path):
        continue

    print(f"Processing {label}...")

    for image_name in os.listdir(label_path):

        image_path = os.path.join(label_path, image_name)

        image = cv2.imread(image_path)

        if image is None:
            failed.append(image_path)
            continue

        rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        result = hands.process(rgb)

        if result.multi_hand_landmarks:

            hand = result.multi_hand_landmarks[0]

            landmarks = []

            for lm in hand.landmark:
                landmarks.extend([lm.x, lm.y, lm.z])

            landmarks.append(label)

            rows.append(landmarks)

        else:
            failed.append(image_path)

# -----------------------------
# Save CSV
# -----------------------------
df = pd.DataFrame(rows)

df.to_csv(OUTPUT_CSV, index=False, header=False)

print("\nDone!")
print("Saved:", OUTPUT_CSV)
print("Samples:", len(rows))
print("Failed:", len(failed))

Processing A...


C:\Users\mania\Downloads\Sign-language-transmittor-main\Sign-language-transmittor-main\venv\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


Processing B...
Processing C...
Processing D...
Processing E...
Processing F...
Processing G...
Processing H...
Processing I...
Processing J...
Processing K...
Processing L...
Processing M...
Processing N...
Processing O...
Processing P...
Processing Q...
Processing R...
Processing S...
Processing T...
Processing U...
Processing V...
Processing W...
Processing X...
Processing Y...
Processing Z...

Done!
Saved: ../data/isl_landmarks.csv
Samples: 23718
Failed: 2284


In [34]:
import pandas as pd

df = pd.read_csv("../data/isl_landmarks.csv", header=None)

print(df.shape)

(23718, 64)


In [36]:
df.head()

,0,1,2,3,4,5,6,7,8,9,...,54,55,56,57,58,59,60,61,62,63
0,0.873381,0.777468,-0.000002,0.741473,0.696254,-0.054544,0.678979,0.536014,-0.073033,0.643308,...,0.955585,0.589210,-0.104226,0.908277,0.648327,-0.087507,0.913983,0.664557,-0.064557,A
1,0.328084,0.897537,-0.000001,0.432478,0.788048,-0.052588,0.458852,0.606444,-0.073075,0.476794,...,0.125739,0.675315,-0.121356,0.187838,0.729205,-0.101492,0.176081,0.733444,-0.081284,A
2,0.330544,0.903060,-0.000001,0.439490,0.793062,-0.052163,0.465156,0.607111,-0.070889,0.484016,...,0.141027,0.678584,-0.120477,0.200464,0.734490,-0.102182,0.184858,0.735873,-0.083329,A
3,0.335072,0.901964,-0.000001,0.438524,0.790893,-0.055317,0.466400,0.609422,-0.077672,0.485416,...,0.139654,0.676593,-0.116323,0.197387,0.731903,-0.097349,0.179455,0.732191,-0.077964,A
4,0.326334,0.899374,-0.000001,0.435938,0.792929,-0.053116,0.468364,0.611097,-0.072466,0.485140,...,0.138617,0.672785,-0.117244,0.197451,0.729235,-0.098182,0.182253,0.730865,-0.078774,A


In [37]:
print(df.iloc[:, -1].unique())

['A' 'B' 'C' 'D' 'E' 'F' 'G' 'H' 'I' 'J' 'K' 'L' 'M' 'N' 'O' 'P' 'Q' 'R'
 'S' 'T' 'U' 'V' 'W' 'X' 'Y' 'Z']


In [35]:
import pandas as pd

df = pd.read_csv("../data/isl_landmarks.csv", header=None)

print(df.iloc[:, -1].value_counts())

63
I    1000
L     999
O     994
R     993
N     993
G     993
J     992
H     992
M     992
V     990
Y     988
A     987
C     984
Q     975
K     958
S     953
B     937
D     928
P     915
Z     861
W     860
F     818
E     744
T     727
U     590
X     555
Name: count, dtype: int64


In [38]:
import os

DATASET_PATH = "../data/ISL"

for folder in sorted(os.listdir(DATASET_PATH)):
    path = os.path.join(DATASET_PATH, folder)

    if os.path.isdir(path):
        print(folder, len(os.listdir(path)))

A 1000
B 1000
C 1000
D 1000
E 1001
F 1000
G 1000
H 1000
I 1000
J 1000
K 1000
L 1000
M 1000
N 1000
O 1000
P 1000
Q 1000
R 1000
S 1000
T 1000
U 1000
V 1000
W 1000
X 1000
Y 1000
Z 1001


In [39]:
import pandas as pd

df = pd.read_csv("../data/isl_landmarks.csv", header=None)

print(df.shape)

print(df.iloc[:, -1].value_counts())

(23718, 64)
63
I    1000
L     999
O     994
R     993
N     993
G     993
J     992
H     992
M     992
V     990
Y     988
A     987
C     984
Q     975
K     958
S     953
B     937
D     928
P     915
Z     861
W     860
F     818
E     744
T     727
U     590
X     555
Name: count, dtype: int64


In [40]:
X = df.iloc[:, :-1].values
y = df.iloc[:, -1].values

print(X.shape)
print(y.shape)

(23718, 63)
(23718,)


In [41]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()
y_encoded = encoder.fit_transform(y)

print("Classes:")
print(encoder.classes_)

Classes:
['A' 'B' 'C' 'D' 'E' 'F' 'G' 'H' 'I' 'J' 'K' 'L' 'M' 'N' 'O' 'P' 'Q' 'R'
 'S' 'T' 'U' 'V' 'W' 'X' 'Y' 'Z']


In [42]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

print(X_train.shape)
print(X_test.shape)

(18974, 63)
(4744, 63)


In [43]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

model = Sequential([
    tf.keras.layers.Input(shape=(63,)),
    Dense(256, activation="relu"),
    Dense(128, activation="relu"),
    Dense(64, activation="relu"),
    Dense(26, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape   ┃ Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━┩
│ dense_4 (Dense)     │ (None, 256)    │  16,384 │
├─────────────────────┼────────────────┼─────────┤
│ dense_5 (Dense)     │ (None, 128)    │  32,896 │
├─────────────────────┼────────────────┼─────────┤
│ dense_6 (Dense)     │ (None, 64)     │   8,256 │
├─────────────────────┼────────────────┼─────────┤
│ dense_7 (Dense)     │ (None, 26)     │   1,690 │
└─────────────────────┴────────────────┴─────────┘

 Total params: 59,226 (231.35 KB)

 Trainable params: 59,226 (231.35 KB)

 Non-trainable params: 0 (0.00 B)

In [44]:
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_test, y_test),
    epochs=20,
    batch_size=128
)

Epoch 1/20
149/149 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.5462 - loss: 1.7695 - val_accuracy: 0.8175 - val_loss: 0.7645
Epoch 2/20
149/149 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.8627 - loss: 0.5487 - val_accuracy: 0.8739 - val_loss: 0.4359
Epoch 3/20
149/149 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9146 - loss: 0.3363 - val_accuracy: 0.9298 - val_loss: 0.2756
Epoch 4/20
149/149 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9392 - loss: 0.2377 - val_accuracy: 0.9418 - val_loss: 0.2021
Epoch 5/20
149/149 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9515 - loss: 0.1823 - val_accuracy: 0.9637 - val_loss: 0.1608
Epoch 6/20
149/149 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9587 - loss: 0.1533 - val_accuracy: 0.9614 - val_loss: 0.1418
Epoch 7/20
149/149 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9619 - loss: 0.1338 - val_accuracy: 0.9361 - val_loss: 0.1953
Epoch 8/20
149/149 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9648 - loss: 0.1210 - val_accuracy: 0.

In [45]:
loss, acc = model.evaluate(X_test, y_test)

print("Test Accuracy:", acc)
print("Test Loss:", loss)

149/149 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9836 - loss: 0.0587
Test Accuracy: 0.983558177947998
Test Loss: 0.05866609513759613


In [47]:
import os
import pickle

# Create model folder if it doesn't exist
os.makedirs("../model", exist_ok=True)

# Save model
model.save("../model/isl_model.h5")

# Save label encoder
with open("../model/isl_label_encoder.pkl", "wb") as f:
    pickle.dump(encoder, f)

print("ISL model saved successfully!")

ISL model saved successfully!


In [48]:
import os

print(os.listdir("../model"))

['asl_label_encoder.pkl', 'asl_model.h5', 'gesture_model.h5', 'isl_label_encoder.pkl', 'isl_model.h5', 'label_encoder.pkl']


In [30]:
import os

DATASET_PATH = "../data/ISL"

for folder in sorted(os.listdir(DATASET_PATH)):
    path = os.path.join(DATASET_PATH, folder)

    if os.path.isdir(path):
        print(folder, len(os.listdir(path)))

A 1000
B 1000
C 1000
D 1000
E 1001
F 1000
G 1000
H 1000
I 1000
J 1000
K 1000
L 1000
M 1000
N 1000
O 1000
P 1000
Q 1000
R 1000
S 1000
T 1000
U 1000
V 1000
W 1000
X 1000
Y 1000
Z 1001
